In [1]:
%run ../scripts/notebook_settings_lean.py
from scipy import stats
from horizonplot import horizonplot
from chromwindow import window
import zarr
import allel
pd.options.display.float_format = '{:10,.3g}'.format 

Abridged notebook based on rfmix10

In [3]:
window_size = 100000
all_chroms = ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX", "chrX", "dipmale_chrX"]
mask_dir = "/home/eriks/primatediversity/people/erik/Primate_Het_X_Autosome/data/callmasks/"

chroms = ["chr{}".format(x) for x in (range(1, 21))] + ["chrX"] #+["all_chrX", "female_chrX", "dipmale_chrX"]

mask_df_l = []
for c in chroms:
    print(c)
    mask_df = pd.read_csv(mask_dir+"cutoff_10_all_baboons_{}_min_third_merged.bed".format(c), sep='\t',
                         comment='t', header=None, names=["chrom", "chromStart", "chromEnd"])
    mask_percentage = []
    for i in range(0, mask_df.chromEnd.iloc[-1], window_size):
        mask_subset = mask_df.loc[(mask_df.chromEnd >= i) & (mask_df.chromStart < i+window_size)]
        start_sum, end_sum = sum(mask_subset.chromStart), sum(mask_subset.chromEnd)
        if mask_subset.chromStart.iloc[0] < i:
            start_sum=start_sum-mask_subset.chromStart.iloc[0]+i
        if mask_subset.chromEnd.iloc[-1] > i+window_size:
            end_sum=end_sum-mask_subset.chromEnd.iloc[-1]+i+window_size
        mask_percentage.append((end_sum-start_sum)/window_size)
    mask_df = pd.DataFrame({"chrom": c, "start": list(range(0, mask_df.chromEnd.iloc[-1], window_size)),
                           "callable_frac": mask_percentage})
    mask_df_l.append(mask_df)
mask_df_all_chroms = pd.concat(mask_df_l)

chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
chrX


In [5]:
import geneinfo as gi
%env ftp_proxy http://proxy-default:3128
%env http_proxy http://proxy-default:3128
%env https_proxy http://proxy-default:3128
gene_df_l = []        

for c in chroms: #["all_chrX"]: #window_df.chrom.unique()[:1]:
    pos_mapping = {}
    print(c)
    chr_df = mask_df_all_chroms.loc[(mask_df_all_chroms.chrom == c)]
    windows = chr_df.start.unique()
    max_window = windows.max()
    genetic_map = "/home/eriks/baboondiversity/data/PG_panu3_recombination_map/mikumi_pyrho_genetic_map_{}.txt"
    if c not in ["all_chrX", "female_chrX", "dipmale_chrX"]:
        chr_recomb = pd.read_csv(genetic_map.format(c), sep=" ")
        chr_genes = gi.get_genes_region(c, 0, max_window, assembly='papAnu4')
    else:
        chr_recomb = pd.read_csv(genetic_map.format(c), sep=" ")
        chr_genes = gi.get_genes_region(c, 0, max_window, assembly='papAnu4')
    for s in chr_df.start.unique():
        if s % 25000000 == 0:
            print(s/max_window)
        cM_sub = chr_recomb.loc[chr_recomb.position >= s]
        if len(cM_sub) > 0:
            cM_pos = chr_recomb.loc[chr_recomb.position >= s]["Genetic_Map(cM)"].iloc[0]
            pos_mapping[s] = cM_pos
        else:
            pos_mapping[s] = chr_recomb["Genetic_Map(cM)"].max()
    chr_df["end"] = list(pos_mapping.keys())[1:]+[chr_recomb.position.max()]
    chr_df["cM"] = chr_df.start.map(pos_mapping)
    pushed_cM = list(pos_mapping.values())[1:]+[chr_recomb["Genetic_Map(cM)"].max()]
    chr_df["end_cM"] = chr_df.start.map(dict(zip(pos_mapping.keys(), pushed_cM)))
    chr_df["average_cM_window"] = (chr_df.end_cM - chr_df.cM) / (chr_df.end - chr_df.start)
    gene_list = []
    gene_pos_mapping = {}
    for g in chr_genes:
        #print(g)
        if g in gene_list or g[0][:3] == "LOC":
            pass
        else:
            gene_list.append(g)
    for w in range(0, max_window+100000, 100000):
        gene_pos_mapping[w] = []
    for g in gene_list:
        s, e = (g[1]//100000)*100000, (g[2]//100000)*100000
        for i in range(s, e+100000, 100000):
            gene_pos_mapping[i].append(g[0])
    chr_df["genes"] = chr_df.start.map(gene_pos_mapping)
    chr_df["genic"] = [True if len(x) > 0 else False for x in chr_df.genes]
    gene_df_l.append(chr_df)
c_r_g_df = pd.concat(gene_df_l)

env: ftp_proxy=http://proxy-default:3128
env: http_proxy=http://proxy-default:3128
env: https_proxy=http://proxy-default:3128
chr1
0.0
0.11499540018399264
0.22999080036798528
0.34498620055197793
0.45998160073597055
0.5749770009199632
0.6899724011039559
0.8049678012879485
0.9199632014719411


/tmp/43602910/ipykernel_626128/1317383782.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chr_df["end"] = list(pos_mapping.keys())[1:]+[chr_recomb.position.max()]
/tmp/43602910/ipykernel_626128/1317383782.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chr_df["cM"] = chr_df.start.map(pos_mapping)
/tmp/43602910/ipykernel_626128/1317383782.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

chr2
0.0
0.13283740701381508
0.26567481402763016
0.3985122210414453
0.5313496280552603
0.6641870350690755
0.7970244420828906
0.9298618490967057


/tmp/43602910/ipykernel_626128/1317383782.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chr_df["end"] = list(pos_mapping.keys())[1:]+[chr_recomb.position.max()]
/tmp/43602910/ipykernel_626128/1317383782.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chr_df["cM"] = chr_df.start.map(pos_mapping)
/tmp/43602910/ipykernel_626128/1317383782.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

chr3
0.0
0.13789299503585217
0.27578599007170435
0.4136789851075565
0.5515719801434087
0.6894649751792609
0.827357970215113
0.9652509652509652


...

0.0
0.3472222222222222
0.6944444444444444


/tmp/43602910/ipykernel_626128/1317383782.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chr_df["end"] = list(pos_mapping.keys())[1:]+[chr_recomb.position.max()]
/tmp/43602910/ipykernel_626128/1317383782.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chr_df["cM"] = chr_df.start.map(pos_mapping)
/tmp/43602910/ipykernel_626128/1317383782.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

chrX
0.0
0.17409470752089137
0.34818941504178275
0.5222841225626741
0.6963788300835655
0.8704735376044568


/tmp/43602910/ipykernel_626128/1317383782.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chr_df["end"] = list(pos_mapping.keys())[1:]+[chr_recomb.position.max()]
/tmp/43602910/ipykernel_626128/1317383782.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chr_df["cM"] = chr_df.start.map(pos_mapping)
/tmp/43602910/ipykernel_626128/1317383782.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

OSError: Cannot save file into a non-existent directory: '../steps/rfmix_stats_df'

In [6]:
c_r_g_df.to_csv("../steps/rfmix_stats_df/call_recomb_genes.txt", index=False)